# Notebook 16 — IAM Benchmark: Pipeline Validation + Domain-Gap Demonstration

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import random, time
from dataclasses import dataclass
from functools import partial
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [2]:
@dataclass
class Config:
    iam_root: Path = Path("../data/iam_words")
    words_txt: str = "words.txt"
    img_subdir: str = "words"
    img_height: int = 48; img_width: int = 320
    subset_size: int = 10000          # representative subset (fixed seed)
    min_label_len: int = 1
    max_label_len: int = 20           # drop pathologically long tokens
    rnn_hidden: int = 256; rnn_layers: int = 2; dropout: float = 0.2
    batch_size: int = 64; epochs: int = 60; lr: float = 1e-4
    warmup_epochs: int = 5; weight_decay: float = 1e-4; grad_clip: float = 5.0
    early_stop_patience: int = 10; num_workers: int = 0
    ckpt_dir: Path = Path("../checkpoints/benchmark_iam")

CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)

## 1. Parse words.txt -> build a label table
Format: `word-id ok graylevel x y w h tag transcription`. We keep only `ok` segmentations,
reconstruct the nested image path, and filter degenerate labels. Then sample the subset.

In [3]:
def parse_words_txt(path, img_root, cfg):
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 9:
                continue
            wid, seg = parts[0], parts[1]
            if seg != "ok":                      # skip bad segmentations
                continue
            transcription = " ".join(parts[8:])  # transcription may contain spaces (rare)
            # reconstruct path: a01-000u-00-00 -> words/a01/a01-000u/a01-000u-00-00.png
            f1 = wid.split("-")[0]               # a01
            f2 = "-".join(wid.split("-")[:2])    # a01-000u
            img_path = img_root / f1 / f2 / f"{wid}.png"
            rows.append({"wid": wid, "path": str(img_path), "label": transcription})
    df = pd.DataFrame(rows)
    # filter degenerate / overly long labels
    df = df[df["label"].str.len().between(cfg.min_label_len, cfg.max_label_len)].reset_index(drop=True)
    return df

img_root = CFG.iam_root / CFG.img_subdir
all_df = parse_words_txt(CFG.iam_root / CFG.words_txt, img_root, CFG)
print(f"parsed {len(all_df)} ok-segmented words")

# keep only rows whose image file actually exists (some segmentations may be missing)
exists = all_df["path"].apply(lambda p: Path(p).exists())
all_df = all_df[exists].reset_index(drop=True)
print(f"{len(all_df)} words with present image files")

# sample representative subset (fixed seed) then split 80/10/10
subset = all_df.sample(n=min(CFG.subset_size, len(all_df)), random_state=SEED).reset_index(drop=True)
n = len(subset); n_tr, n_va = int(0.8 * n), int(0.1 * n)
train_df = subset.iloc[:n_tr].reset_index(drop=True)
val_df = subset.iloc[n_tr:n_tr + n_va].reset_index(drop=True)
test_df = subset.iloc[n_tr + n_va:].reset_index(drop=True)
print(f"subset={n} -> train={len(train_df)} val={len(val_df)} test={len(test_df)}")
print(f"sample labels: {train_df['label'].head(8).tolist()}")

parsed 38305 ok-segmented words
38305 words with present image files
subset=10000 -> train=8000 val=1000 test=1000
sample labels: ['a', ',', 'industry', 'degree', 'part', 'are', 'parties', 'of']


## 2. Vocab / Dataset / CRNN / metrics (same architecture as the prescription baseline)
IAM has a larger character set (upper+lower+digits+punctuation) than prescriptions — the CRNN
handles it; only the number of output classes changes.

In [4]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self): return len(self.idx2char) + 1
    def encode(self, t): return [self.char2idx[c] for c in t if c in self.char2idx]
    def decode_greedy(self, indices):
        out, prev = [], None
        for i in indices:
            if i != prev and i != self.BLANK:
                out.append(self.idx2char.get(i, ""))
            prev = i
        return "".join(out)

class IAMDataset(Dataset):
    def __init__(self, df, cfg, vocab=None, augment=False):
        self.df = df; self.cfg = cfg; self.vocab = vocab; self.augment = augment
    def labels(self): return self.df["label"].tolist()
    def __len__(self): return len(self.df)
    def _load(self, path):
        img = Image.open(path).convert("L"); w, h = img.size
        if w < 1 or h < 1:
            img = Image.new("L", (self.cfg.img_width, self.cfg.img_height), 255)
            w, h = img.size
        nw = min(max(1, int(round(w * self.cfg.img_height / h))), self.cfg.img_width)
        img = img.resize((nw, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), 255); canvas.paste(img, (0, 0))
        return canvas
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = self._load(row["path"])
        except Exception:
            img = Image.new("L", (self.cfg.img_width, self.cfg.img_height), 255)
        if self.augment:
            img = img.rotate(random.uniform(-3, 3), resample=Image.BILINEAR, fillcolor=255)
        x = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
        target = torch.tensor(self.vocab.encode(row["label"]), dtype=torch.long)
        return x, target, row["label"], row["wid"]

def collate(batch):
    xs, targets, texts, ids = zip(*batch)
    tl = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    return torch.stack(xs), torch.cat(targets), tl, list(texts), list(ids)

def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

class CRNN(nn.Module):
    def __init__(self, n_classes, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(*conv(1,64), nn.MaxPool2d(2,2), *conv(64,128), nn.MaxPool2d(2,2),
            *conv(128,256), *conv(256,256), nn.MaxPool2d((2,1),(2,1)),
            *conv(256,512,bn=True), *conv(512,512,bn=True), nn.MaxPool2d((2,1),(2,1)))
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2*rnn_hidden, n_classes)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f); return self.head(seq)

## 3. Data loaders

In [5]:
train_ds = IAMDataset(train_df, CFG, augment=True)
VOCAB = Vocab(train_ds.labels()); train_ds.vocab = VOCAB
val_ds = IAMDataset(val_df, CFG, vocab=VOCAB); test_ds = IAMDataset(test_df, CFG, vocab=VOCAB)
print(f"IAM vocab size: {len(VOCAB)} (chars: {''.join(list(VOCAB.char2idx)[:40])}...)")

train_dl = DataLoader(train_ds, CFG.batch_size, shuffle=True, num_workers=CFG.num_workers,
                      collate_fn=collate, drop_last=True)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

IAM vocab size: 76 (chars:  !"#'(),-./0123456789:;?ABCDEFGHIJKLMNOP...)


## 4. Train (CTC), early stopping

In [6]:
def greedy(model, loader):
    model.eval(); preds, refs = [], []
    with torch.no_grad():
        for xb, _, _, texts, _ in loader:
            logits = model(xb.to(DEVICE)); idx = logits.argmax(-1).permute(1, 0).cpu()
            preds += [VOCAB.decode_greedy(s.tolist()) for s in idx]; refs += texts
    return preds, refs

def train(model):
    model.to(DEVICE)
    ctc = nn.CTCLoss(blank=Vocab.BLANK, zero_infinity=True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    best, no_imp = float("inf"), 0
    for epoch in range(1, CFG.epochs + 1):
        if epoch <= CFG.warmup_epochs:
            for g in opt.param_groups: g["lr"] = CFG.lr * epoch / CFG.warmup_epochs
        model.train(); t0 = time.time()
        for xb, targets, tlens, _, _ in train_dl:
            xb = xb.to(DEVICE); logits = model(xb); T = logits.shape[0]
            loss = ctc(logits.log_softmax(2).cpu(), targets,
                       torch.full((xb.shape[0],), T, dtype=torch.long), tlens)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); opt.step()
        vp, vr = greedy(model, val_dl); vm = corpus_metrics(vp, vr)
        if epoch > CFG.warmup_epochs: sched.step(vm["CER"])
        if epoch % 5 == 0 or epoch == 1:
            print(f"epoch {epoch:3d} | val CER {vm['CER']:.4f} | val EM {vm['ExactMatch']:.4f} | {time.time()-t0:.1f}s")
        if vm["CER"] < best:
            best, no_imp = vm["CER"], 0
            torch.save({"model": model.state_dict(), "vocab": VOCAB.idx2char}, CFG.ckpt_dir / "best.pt")
        else:
            no_imp += 1
            if no_imp >= CFG.early_stop_patience:
                print(f"early stop at epoch {epoch} (best val CER {best:.4f})"); break

model = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
train(model)

params: 7.69M
epoch   1 | val CER 1.0000 | val EM 0.0000 | 42.4s
epoch   5 | val CER 0.8328 | val EM 0.1400 | 38.1s
epoch  10 | val CER 0.8005 | val EM 0.0620 | 38.4s
epoch  15 | val CER 0.5963 | val EM 0.2680 | 38.1s
epoch  20 | val CER 0.3398 | val EM 0.4080 | 38.2s
epoch  25 | val CER 0.2412 | val EM 0.4980 | 39.0s
epoch  30 | val CER 0.2311 | val EM 0.5030 | 37.9s
epoch  35 | val CER 0.2197 | val EM 0.5150 | 37.8s
epoch  40 | val CER 0.1850 | val EM 0.5910 | 39.0s
epoch  45 | val CER 0.1770 | val EM 0.5940 | 39.1s
epoch  50 | val CER 0.1684 | val EM 0.6100 | 38.6s
epoch  55 | val CER 0.1733 | val EM 0.5990 | 38.4s
epoch  60 | val CER 0.1731 | val EM 0.5980 | 38.8s
early stop at epoch 60 (best val CER 0.1684)


## 5. Test + the domain-gap comparison

In [7]:
ck = torch.load(CFG.ckpt_dir / "best.pt", map_location="cpu")
model.load_state_dict(ck["model"]); model.to(DEVICE)
tp, tr = greedy(model, test_dl); tm = corpus_metrics(tp, tr)
print(f"IAM — test (n={tm['n']}): CER {tm['CER']:.4f} | EM {tm['ExactMatch']:.4f}")

print(f"""
DOMAIN-GAP COMPARISON (same CRNN architecture):
  IAM (general English words)     : CER {tm['CER']:.3f}   <- this run
  Prescription custom (open-vocab): CER ~0.45            (from baseline)
  Prescription BD (closed-vocab)  : CER ~0.15            (from Notebook 10)

INTERPRETATION (for the thesis):
- If IAM CER is clearly LOWER (better) than the custom prescription CER, the same architecture
  reads general handwriting more accurately than open-vocabulary prescriptions -> the prescription
  domain is genuinely harder. This validates the pipeline AND motivates the domain-specific
  contributions (lexicon, fusion). State it as: "our pipeline is competent on the standard IAM
  benchmark; the elevated prescription error reflects domain difficulty (rare/illegible medicine
  names, open vocabulary), not a deficiency of the method."
- Do NOT claim to beat published IAM systems — different subset/protocol. Claim competence + gap.
""")

pd.DataFrame([{"dataset": "IAM-subset", "CER": tm["CER"], "EM": tm["ExactMatch"], "n": tm["n"],
               "subset_size": CFG.subset_size}]).to_csv(CFG.ckpt_dir / "iam_results.csv", index=False)

IAM — test (n=1000): CER 0.1891 | EM 0.5610

DOMAIN-GAP COMPARISON (same CRNN architecture):
  IAM (general English words)     : CER 0.189   <- this run
  Prescription custom (open-vocab): CER ~0.45            (from baseline)
  Prescription BD (closed-vocab)  : CER ~0.15            (from Notebook 10)

INTERPRETATION (for the thesis):
- If IAM CER is clearly LOWER (better) than the custom prescription CER, the same architecture
  reads general handwriting more accurately than open-vocabulary prescriptions -> the prescription
  domain is genuinely harder. This validates the pipeline AND motivates the domain-specific
  contributions (lexicon, fusion). State it as: "our pipeline is competent on the standard IAM
  benchmark; the elevated prescription error reflects domain difficulty (rare/illegible medicine
  names, open vocabulary), not a deficiency of the method."
- Do NOT claim to beat published IAM systems — different subset/protocol. Claim competence + gap.



## 6. Sample predictions (qualitative)

In [8]:
print("sample IAM predictions (pred | truth):")
for p, r in list(zip(tp, tr))[:15]:
    mark = "OK " if p == r else "  x"
    print(f"  {mark} {p!r:22} | {r!r}")

sample IAM predictions (pred | truth):
    x 'af.'                  | 'af-'
    x 'cownot'               | 'cannot'
  OK  'in.'                  | 'in.'
    x 'scesfolly'            | 'successfully'
    x 'siugers'              | 'singers'
  OK  '.'                    | '.'
    x 'hers'                 | 'the'
    x 'mucrass'              | 'success'
  OK  '.'                    | '.'
  OK  'burdens'              | 'burdens'
  OK  'the'                  | 'the'
    x 'hae'                  | 'have'
    x 'setalintuy'           | 'retaliatory'
    x 'preesrare'            | 'pressure'
    x 'wpsetting'            | 'upsetting'
